In [1]:
import pathlib
import pandas as pd

features_path = pathlib.Path('/data/road_surface_classifier/features.csv')
df = pd.read_csv(features_path).iloc[:, 1:].set_index('osm_id')
df = df.drop(columns=['x', 'y', 'ix', 'iy', 'length', 'dist_sq'])

In [2]:
# Create in-mem dataset of points associated with OSM id

from osgeo import ogr, osr

# Create SRS (EPSG:4326: WGS-84 decimal degrees)
srs = osr.SpatialReference()
srs.ImportFromEPSG(4326)

driver: ogr.Driver = ogr.GetDriverByName('Memory')
ds: ogr.DataSource = driver.CreateDataSource('/vsimem/tmp_features')
layer: ogr.Layer = ds.CreateLayer('data', srs=srs, geom_type=ogr.wkbPoint)

osm_id_field = ogr.FieldDefn('osm_id', ogr.OFTInteger64)

layer.CreateField(osm_id_field)

feature_defn = layer.GetLayerDefn()

for _, row in df.iterrows():
    feat = ogr.Feature(feature_defn)

    pt = ogr.Geometry(ogr.wkbPoint)
    pt.AddPoint_2D(row['lon'], row['lat'])

    feat.SetGeometry(pt)
    feat.SetField('osm_id', row.name)
    layer.CreateFeature(feat)
    pt = None
    feat = None

In [3]:
df['naip_searched'] = False
for new_col in ('naip_entity_id', 'naip_display_id', 'naip_overlay_url'):
    df[new_col] = None

df

,highway,surface,wkt,lon,lat,naip_searched,naip_entity_id,naip_display_id,naip_overlay_url
osm_id,,,,,,,,,
2044,tertiary,asphalt,"LINESTRING (-116.1637003 33.9926953,-116.16372...",-116.163886,34.003964,False,None,None,None
210764650,tertiary,asphalt,"LINESTRING (-78.6055292 35.86147,-78.6055276 3...",-78.605339,35.862812,False,None,None,None
210764651,tertiary,asphalt,"LINESTRING (-78.606587 35.872842,-78.6065755 3...",-78.606065,35.876237,False,None,None,None
210764652,primary,asphalt,"LINESTRING (-78.593971 35.8121115,-78.5941294 ...",-78.597025,35.811999,False,None,None,None
210764661,residential,asphalt,"LINESTRING (-78.6095782 35.8674006,-78.6096618...",-78.612629,35.865788,False,None,None,None
...,...,...,...,...,...,...,...,...,...
323162813,unclassified,unpaved,"LINESTRING (-112.806823 46.6216025,-112.806594...",-112.784251,46.608616,False,None,None,None
323162814,residential,unpaved,"LINESTRING (-112.7830251 46.6035267,-112.78285...",-112.764100,46.626201,False,None,None,None
323004406,residential,unpaved,"LINESTRING (-112.5128724 35.085824,-112.512582...",-112.501002,35.092799,False,None,None,None


In [4]:
from typing import Optional

from osgeo import ogr

from usgs_api.naip import NAIPSceneSearchRequest, NAIPSceneSearchResult
from usgs_api.data_types import SpatialFilterMbr, Coordinate, SceneFilter

def naip_search_point(lat: float, lon: float) -> Optional[NAIPSceneSearchResult]:
    # Build search envelope
    TOL = 0.00001
    ll_lat, ll_lon = lat - TOL, lon - TOL
    ur_lat, ur_lon = lat + TOL, lon + TOL

    # Build query filter
    spatial_filter = SpatialFilterMbr(lower_left=Coordinate(ll_lat, ll_lon), upper_right=Coordinate(ur_lat, ur_lon))
    scene_filter = SceneFilter(spatial_filter=spatial_filter)

    # Perform query
    results = []
    total_hits = 1
    while len(results) < total_hits:
        request = NAIPSceneSearchRequest(scene_filter=scene_filter, starting_number=len(results) + 1, max_results=1000)
        response = request.make_request()
        total_hits = response.data['totalHits']
        results.extend(response.data['results'])
        if len(results) < total_hits:
            print(f'Got {len(results):d} of {total_hits:d} results. Requesting more...')

    # Raise error if no resu
    if not len(results):
        return

    # Convert to NAIP results, and return the one with most recent acquisition time
    results_naip = [NAIPSceneSearchResult.from_dict(d) for d in results]
    return sorted(results_naip, key = lambda r: r.acq_time, reverse=True)[0]



In [28]:
# This is the hard part. Do the thing!
from traceback import print_exc

counter = 0
while len(df[df.naip_searched == False]):
    # Get first unsearched row and mark it so we don't search it again
    this_row = df[df.naip_searched == False].iloc[0, :]
    df.loc[this_row.name, 'naip_searched'] = True  # type: ignore

    # Print status
    counter += 1
    if counter % 100 == 0:
        print(f'Performed {counter} searches. Maximum {len(df[df.naip_searched == False])} to go.')

    try:
        result = naip_search_point(this_row['lat'], this_row['lon'])
        assert result is not None
    except Exception as e:
        print_exc()
        print('Got exception during `naip_search_point` osm_id %s: %s' % (this_row.name, str(e)))
        continue

    # Filter any objects that are found in the result's spatial coverage, and set the corresponding info
    try:
        layer.SetSpatialFilter(result.spatial_coverage())
        for _ in range(layer.GetFeatureCount()):
            feat: ogr.Feature = layer.GetNextFeature()
            osm_id = feat.GetFieldAsInteger('osm_id')
            df.loc[osm_id, 'naip_searched'] = True
            df.loc[osm_id, 'naip_entity_id'] = result.entity_id
            df.loc[osm_id, 'naip_display_id'] = result.display_id
            df.loc[osm_id, 'naip_overlay_url'] = result.overlay_url
    except Exception as e:
        print('Got exception during filtered search for osm_id %s: %s' % (this_row.name, str(e)))
        continue

Traceback (most recent call last):
  File "/tmp/ipykernel_126124/2830103727.py", line 17, in <cell line: 5>
    assert result is not None
AssertionError


Got exception during `naip_search_point` osm_id 233967784: 


Traceback (most recent call last):
  File "/tmp/ipykernel_126124/2830103727.py", line 17, in <cell line: 5>
    assert result is not None
AssertionError


Got exception during `naip_search_point` osm_id 195260803: 


Traceback (most recent call last):
  File "/tmp/ipykernel_126124/2830103727.py", line 17, in <cell line: 5>
    assert result is not None
AssertionError


...

Traceback (most recent call last):
  File "/tmp/ipykernel_126124/2830103727.py", line 17, in <cell line: 5>
    assert result is not None
AssertionError


Got exception during `naip_search_point` osm_id 338599238: 


Traceback (most recent call last):
  File "/tmp/ipykernel_126124/2830103727.py", line 17, in <cell line: 5>
    assert result is not None
AssertionError


Got exception during `naip_search_point` osm_id 338599240: 
Got exception during `naip_search_point` osm_id 338599242: 


Traceback (most recent call last):
  File "/tmp/ipykernel_126124/2830103727.py", line 17, in <cell line: 5>
    assert result is not None
AssertionError


In [30]:
len(df[df.naip_entity_id.isnull()])



36

In [24]:
import usgs_api
usgs_api.get_api_key(reset=True)

2022-11-27 20:11:42,909 INFO [api_key] Fetching API key...


'eyJjaWQiOjI2MjQ4MDU4LCJzIjoiMTY2OTYwNTEwMyIsInIiOjY5OSwicCI6WyJ1c2VyIiwiZG93bmxvYWQiLCJvcmRlciJdfQ=='

In [31]:
df.to_csv('/data/road_surface_classifier/jon_did_it.csv')
df.to_pickle('/data/road_surface_classifier/jon_did_it.pkl')
df

,highway,surface,wkt,lon,lat,naip_searched,naip_entity_id,naip_display_id,naip_overlay_url
osm_id,,,,,,,,,
2044,tertiary,asphalt,"LINESTRING (-116.1637003 33.9926953,-116.16372...",-116.163886,34.003964,True,2992288,M_3411663_SE_11_060_20200416,https://ims.cr.usgs.gov/browse/naip/fullres/CA...
210764650,tertiary,asphalt,"LINESTRING (-78.6055292 35.86147,-78.6055276 3...",-78.605339,35.862812,True,3057185,M_3507812_NW_17_060_20200712,https://ims.cr.usgs.gov/browse/naip/fullres/NC...
210764651,tertiary,asphalt,"LINESTRING (-78.606587 35.872842,-78.6065755 3...",-78.606065,35.876237,True,3057156,M_3507804_SW_17_060_20200712,https://ims.cr.usgs.gov/browse/naip/fullres/NC...
210764652,primary,asphalt,"LINESTRING (-78.593971 35.8121115,-78.5941294 ...",-78.597025,35.811999,True,3057187,M_3507812_SW_17_060_20200712,https://ims.cr.usgs.gov/browse/naip/fullres/NC...
210764661,residential,asphalt,"LINESTRING (-78.6095782 35.8674006,-78.6096618...",-78.612629,35.865788,True,3057185,M_3507812_NW_17_060_20200712,https://ims.cr.usgs.gov/browse/naip/fullres/NC...
...,...,...,...,...,...,...,...,...,...
323162813,unclassified,unpaved,"LINESTRING (-112.806823 46.6216025,-112.806594...",-112.784251,46.608616,True,3077186,M_4611226_NE_12_060_20210628,https://ims.cr.usgs.gov/browse/naip/fullres/MT...
323162814,residential,unpaved,"LINESTRING (-112.7830251 46.6035267,-112.78285...",-112.764100,46.626201,True,3077186,M_4611226_NE_12_060_20210628,https://ims.cr.usgs.gov/browse/naip/fullres/MT...
323004406,residential,unpaved,"LINESTRING (-112.5128724 35.085824,-112.512582...",-112.501002,35.092799,True,3106569,M_3511261_NW_12_060_20211031,https://ims.cr.usgs.gov/browse/naip/fullres/AZ...


In [32]:
len(df.naip_entity_id.unique())

16360